## Reddit Mental Health Discussions - Exploratory Data Analysis (EDA)

### This script performs exploratory data analysis (EDA) on a dataset containing Reddit posts related to mental health.

Sections:
1. Load Dataset
2. Data Cleaning & Preprocessing
3. Text Processing (Tokenization, Lemmatization, Stemming)
4. Word Cloud Visualization
5. Subreddit-Specific Word Clouds
6. Medication Mentions Analysis
7. Time Series Analysis
8. User Engagement & Subreddit Activity Analysis

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import nltk
import string
import spacy
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from wordcloud import WordCloud
from collections import Counter

In [ ]:
# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('stopwords')

In [ ]:
# Load spaCy model for lemmatization
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Load Dataset
file_path = "/content/reddit_posts_combined_26Jan_16Mar.csv"
df = pd.read_csv(file_path)
print("Dataset Loaded Successfully")

In [ ]:
# Display dataset information
df.info()
df.head()

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Fill missing values
df.dropna(subset=['Selftext'], inplace=True)
# Redact author handles in the public notebook. Keep only aggregate analysis.
df['Author'] = 'Redacted'
print("Missing values handled and author handles redacted")


In [ ]:
# Drop the 'Created_UTC' column
if 'Created_UTC' in df.columns:
    df = df.drop('Created_UTC', axis=1)
    print("'Created_UTC' column dropped successfully.")
else:
    print("'Created_UTC' column not found in the DataFrame.")

In [ ]:
# Save cleaned dataset
cleaned_file_path = "/content/reddit_posts_26Jan-16Mar_cleaned.csv"
df.to_csv(cleaned_file_path, index=False)
print(f"Cleaned dataset saved at: {cleaned_file_path}")

In [ ]:
# Check for missing values
df.isnull().sum()

### 2️⃣ Text Processing (Tokenization, Lemmatization, Stemming)

In [ ]:
nltk.download('punkt_tab')

In [ ]:
def clean_and_tokenize(text):
    """Tokenize and clean text by removing stopwords and punctuation."""
    if pd.isna(text):
        return ""
    text = text.lower()
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
    return " ".join(tokens)

df["New_Token_Selftext"] = df["Selftext"].apply(clean_and_tokenize)

In [ ]:
def lemmatize_text(text):
    """Perform lemmatization on the text using spaCy."""
    if isinstance(text, str):
        doc = nlp(text)
        return " ".join([token.lemma_ for token in doc])
    return text

df['Lemmatized_Text'] = df['New_Token_Selftext'].apply(lemmatize_text)
df['Merged_Lemma_Token'] = df['Lemmatized_Text']

df.to_csv("/content/reddit_posts_processed_16Mar.csv", index=False)

### 3️⃣ Word Cloud Visualization

In [ ]:
def generate_wordcloud(text, title):
    """Generate and display a word cloud."""
    wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='viridis').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title(title, fontsize=14)
    plt.show()

In [ ]:
# Generate word clouds for different processed text columns
generate_wordcloud(" ".join(df['New_Token_Selftext'].dropna()), "Word Cloud for Tokenized Selftext")
generate_wordcloud(" ".join(df['Lemmatized_Text'].dropna()), "Word Cloud for Lemmatized Text")
generate_wordcloud(" ".join(df['Merged_Lemma_Token'].dropna()), "Word Cloud for Merged Lemma Token")

### 4️⃣ Subreddit-Specific Word Clouds

In [ ]:
subreddits = ["Anxiety", "depression", "ADHD"]
for sub in subreddits:
    sub_data = df[df['Subreddit'] == sub]
    text_data = " ".join(sub_data['Merged_Lemma_Token'].dropna())
    if text_data.strip():
        generate_wordcloud(text_data, f"Word Cloud for {sub}")
    else:
        print(f"No text data available for {sub} subreddit.")

### 5️⃣ Medication Mentions Analysis

In [ ]:
anxiety_medication_list = {"lexapro", "zoloft", "sertraline", "escitalopram", "xanax", "buspirone", "propranolol", "ssri", "prozac", "buspar",
                           "fluoxetine", "ativan", "lorazepam"}
medication_counts = Counter()

for text in df["New_Token_Selftext"].dropna():
    tokens = word_tokenize(text.lower())
    filtered_meds = [word for word in tokens if word in anxiety_medication_list]
    medication_counts.update(filtered_meds)

generate_wordcloud(" ".join(medication_counts.keys()), "Word Cloud - Anxiety Medications")

In [ ]:
depression_medication_list = {"prozac", "zoloft", "sertraline", "fluoxetine", "wellbutrin", "escitalopram", "bupropion", "trazodone", "trimipramine",
                              "doxepin", "effexor", "pristiq", "cymbalta", "lexapro", "celexa", "silenor"}
medication_counts = Counter()

for text in df["New_Token_Selftext"].dropna():
    tokens = word_tokenize(text.lower())
    filtered_meds = [word for word in tokens if word in depression_medication_list]
    medication_counts.update(filtered_meds)

generate_wordcloud(" ".join(medication_counts.keys()), "Word Cloud - Depression Medications")

In [ ]:
adhd_medication_list = {"strattera", "ritalin", "adderall", "vyvanse", "methylphenidate", "bupropion", "dextroamphetamine", "guanfacine",
                        "concerta", "lisdexamfetamine", "dexedrine", "wellbutrin"}
medication_counts = Counter()

for text in df["New_Token_Selftext"].dropna():
    tokens = word_tokenize(text.lower())
    filtered_meds = [word for word in tokens if word in adhd_medication_list]
    medication_counts.update(filtered_meds)

generate_wordcloud(" ".join(medication_counts.keys()), "Word Cloud - ADHD Medications")

In [ ]:
medication_list = {"prozac", "fluoxetine", "lexapro", "escitalopram", "zoloft", "sertraline", "paxil", "paroxetine",
    "celexa", "citalopram", "wellbutrin", "bupropion", "effexor", "venlafaxine", "cymbalta", "duloxetine",
    "abilify", "aripiprazole", "risperdal", "risperidone", "lamictal", "lamotrigine", "vyvanse", "lisdexamfetamine",
    "adderall", "amphetamine", "ritalin", "methylphenidate", "concerta", "strattera", "atomoxetine",
    "xanax", "alprazolam", "klonopin", "clonazepam", "ativan", "lorazepam", "valium", "diazepam",
    "seroquel", "quetiapine", "lithium", "buspar", "buspirone", "propranolol", "ssri"}
medication_counts = Counter()

for text in df["New_Token_Selftext"].dropna():
    tokens = word_tokenize(text.lower())
    filtered_meds = [word for word in tokens if word in medication_list]
    medication_counts.update(filtered_meds)

generate_wordcloud(" ".join(medication_counts.keys()), "Word Cloud - Medications")

### 6️⃣ Time Series Analysis

In [ ]:
# Convert to datetime format
df['Created_DateTime'] = pd.to_datetime(df['Created_DateTime'])
df['Date'] = df['Created_DateTime'].dt.date

daily_posts = df.groupby('Date').size()
plt.figure(figsize=(12, 5))
plt.plot(daily_posts.index, daily_posts.values, marker='o', linestyle='-')
plt.xlabel("Date")
plt.ylabel("Number of Posts")
plt.title("Daily Post Frequency Over Time")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

### 7️⃣ User Engagement & Subreddit Activity Analysis

In [ ]:
# Top 10 most active subreddits
subreddit_counts = df['Subreddit'].value_counts().head(10)
subreddit_counts.plot(kind='bar', color='skyblue', figsize=(12,5))
plt.xlabel("Subreddit")
plt.ylabel("Number of Posts")
plt.title("Top 10 Most Active Subreddits")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--')
plt.show()

In [ ]:
# Aggregate engagement by subreddit. Avoid displaying post titles or author handles.
engagement_by_subreddit = (
    df.groupby('Subreddit')['Score']
    .agg(post_count='count', average_score='mean', total_score='sum')
    .sort_values('total_score', ascending=False)
)
display(engagement_by_subreddit)


In [ ]:
# Post volume by subreddit.
subreddit_counts = df['Subreddit'].value_counts()
subreddit_counts.plot(kind='bar', color='purple', figsize=(10, 5))
plt.xlabel("Subreddit")
plt.ylabel("Number of Posts")
plt.title("Post Volume by Subreddit")
plt.xticks(rotation=30)
plt.grid(axis='y', linestyle='--')
plt.show()


In [ ]:
# Aggregate engagement by subreddit.
subreddit_engagement = df.groupby('Subreddit')['Score'].sum().sort_values(ascending=False)
subreddit_engagement.plot(kind='bar', color='red', figsize=(10, 5))
plt.xlabel("Subreddit")
plt.ylabel("Total Engagement Score")
plt.title("Aggregate Engagement by Subreddit")
plt.xticks(rotation=30)
plt.grid(axis='y', linestyle='--')
plt.show()


In [ ]:
print("EDA Completed Successfully!")

## Sai Aashish Code

In [ ]:
import pandas as pd
from nltk import ngrams
from collections import Counter
import matplotlib.pyplot as plt

In [ ]:
def get_ngrams(text, n):
    if pd.isna(text):  # Check for NaN
        return []
    if isinstance(text, str):  # Ensure text is a string
        words = text.split()
        return list(ngrams(words, n))
    return []  # Return empty list for non-string types

In [ ]:
# Dictionary to store n-gram frequencies by subreddit
bigram_freq_by_subreddit = {}
trigram_freq_by_subreddit = {}

In [ ]:
# Extract bigrams and trigrams by subreddit
for subreddit in df['Subreddit'].unique():
    subreddit_df = df[df['Subreddit'] == subreddit]
    all_bigrams = []
    all_trigrams = []
    for text in subreddit_df['New_Token_Selftext']:
        bigrams = get_ngrams(text, 2)
        trigrams = get_ngrams(text, 3)
        all_bigrams.extend(bigrams)
        all_trigrams.extend(trigrams)
    bigram_freq_by_subreddit[subreddit] = Counter(all_bigrams)
    trigram_freq_by_subreddit[subreddit] = Counter(all_trigrams)

In [ ]:
# Convert to DataFrames and get top 10 for each subreddit
bigram_dfs = {}
trigram_dfs = {}

In [ ]:
for subreddit in df['Subreddit'].unique():
    bigram_df = pd.DataFrame(bigram_freq_by_subreddit[subreddit].items(), columns=['Bigram', 'Frequency'])
    bigram_df['Bigram'] = bigram_df['Bigram'].apply(lambda x: ' '.join(x))  # Convert tuple to string
    bigram_df = bigram_df.sort_values(by=['Frequency', 'Bigram'], ascending=[False, True]).head(10).reset_index(drop=True)
    bigram_dfs[subreddit] = bigram_df

    trigram_df = pd.DataFrame(trigram_freq_by_subreddit[subreddit].items(), columns=['Trigram', 'Frequency'])
    trigram_df['Trigram'] = trigram_df['Trigram'].apply(lambda x: ' '.join(x))  # Convert tuple to string
    trigram_df = trigram_df.sort_values(by=['Frequency', 'Trigram'], ascending=[False, True]).head(10).reset_index(drop=True)
    trigram_dfs[subreddit] = trigram_df

In [ ]:
# Display the tables
for subreddit in df['Subreddit'].unique():
    print(f"\nTop 10 Bigrams Table for {subreddit}:")
    print(bigram_dfs[subreddit].to_string(index=False))
    print(f"\nTop 10 Trigrams Table for {subreddit}:")
    print(trigram_dfs[subreddit].to_string(index=False))

In [ ]:
# Generate bar plots for each subreddit
for subreddit in df['Subreddit'].unique():
    plt.figure(figsize=(15, 5))

    # Bar plot for bigrams
    plt.subplot(1, 2, 1)
    plt.bar(bigram_dfs[subreddit]['Bigram'], bigram_dfs[subreddit]['Frequency'])
    plt.title(f'Top 10 Bigrams - {subreddit}')
    plt.xticks(rotation=45, ha='right')
    plt.xlabel('Bigrams')
    plt.ylabel('Frequency')

    # Bar plot for trigrams
    plt.subplot(1, 2, 2)
    plt.bar(trigram_dfs[subreddit]['Trigram'], trigram_dfs[subreddit]['Frequency'])
    plt.title(f'Top 10 Trigrams - {subreddit}')
    plt.xticks(rotation=45, ha='right')
    plt.xlabel('Trigrams')
    plt.ylabel('Frequency')

    plt.tight_layout()
    plt.show()

### New EDA - Sample

In [ ]:
### 9️⃣ Top 10 - Check the Disease (Post Frequency by Date and Disease Category)
# Convert to datetime format
df['Created_DateTime'] = pd.to_datetime(df['Created_DateTime'])
df['Date'] = df['Created_DateTime'].dt.date

daily_posts = df.groupby('Date').size()
plt.figure(figsize=(12, 5))
plt.plot(daily_posts.index, daily_posts.values, marker='o', linestyle='-')
plt.xlabel("Date")
plt.ylabel("Number of Posts")
plt.title("Daily Post Frequency Over Time")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

In [ ]:
# Check disease category-wise post frequency
disease_posts = df.groupby(['Date', 'Subreddit']).size().unstack().fillna(0)
disease_posts.plot(kind='line', figsize=(12, 6), marker='o')
plt.xlabel("Date")
plt.ylabel("Number of Posts")
plt.title("Daily Post Frequency by Disease Category")
plt.xticks(rotation=45)
plt.legend(title='Subreddit')
plt.grid(True)
plt.show()

In [ ]:
# Check disease category-wise post frequency
disease_posts = df.groupby(['Date', 'Subreddit']).size().unstack().fillna(0)

# Generate separate plots for each disease category
for subreddit in disease_posts.columns:
    plt.figure(figsize=(12, 6))
    plt.plot(disease_posts.index, disease_posts[subreddit], marker='o', linestyle='-')
    plt.xlabel("Date")
    plt.ylabel("Number of Posts")
    plt.title(f"Daily Post Frequency for {subreddit}")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()

In [ ]:
import seaborn as sns # Import the seaborn library

### Author-Level Analysis Omitted

The original class notebook explored top authors. The public version omits author-level outputs to avoid exposing handles from sensitive mental-health discussions.


### Author-Level Analysis Omitted

The original class notebook explored top authors. The public version omits author-level outputs to avoid exposing handles from sensitive mental-health discussions.


### Author-Level Analysis Omitted

The original class notebook explored top authors. The public version omits author-level outputs to avoid exposing handles from sensitive mental-health discussions.
